# Module 1 — Data Pipeline

Scrape live product data from [books.toscrape.com](http://books.toscrape.com/),
clean it into properly typed columns, convert price to INR using a fixed
project-defined baseline rate, load it into a normalized SQLite schema, and
query it with both SQL and pandas.

Pipeline: **scrape → clean → convert → store → query**.

In [1]:
import re
import sqlite3

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://books.toscrape.com/"
DB_PATH = "books.db"

# Fixed, project-defined baseline conversion rate (assignment spec, not a
# live/historical market rate): 1 GBP = 105.50 INR.
GBP_TO_INR = 105.50

# Scrape whole categories rather than a fixed page count, so every book in
# the chosen categories ends up in the dataset (task only requires >=3
# categories and >=60 books).
NUM_CATEGORIES = 5


def get_soup(url):
    """Fetch a page and parse it, raising a clear error on any HTTP failure."""
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return BeautifulSoup(resp.text, "html.parser")

/Users/shellzero/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Scrape

Walk the category sidebar on the homepage, then paginate through each
selected category's listing pages (categories like *Sequential Art* have
more than 20 books and span multiple pages) collecting the raw fields for
every book: `title`, `price` (as listed, GBP), `star_rating` (as text),
`availability` (as listed text), and `category`.

In [2]:
soup = get_soup(BASE_URL + "index.html")
category_links = soup.select("div.side_categories ul.nav-list ul li a")
categories = [(a.text.strip(), BASE_URL + a["href"]) for a in category_links]

selected_categories = categories[:NUM_CATEGORIES]
print("Scraping categories:", [name for name, _ in selected_categories])

records = []
for cat_name, cat_url in selected_categories:
    url = cat_url
    while url:
        soup = get_soup(url)
        for book in soup.select("article.product_pod"):
            records.append({
                "title": book.select_one("h3 a")["title"],
                "price": book.select_one("p.price_color").text,
                "star_rating": book.select_one("p.star-rating")["class"][1],
                "availability": book.select_one("p.instock.availability").text.strip(),
                "category": cat_name,
            })
        # Follow "next" until a category's last listing page stops offering one.
        next_link = soup.select_one("li.next a")
        url = url.rsplit("/", 1)[0] + "/" + next_link["href"] if next_link else None

df_raw = pd.DataFrame(records)
print(f"Scraped {len(df_raw)} books across {df_raw['category'].nunique()} categories")
df_raw["category"].value_counts()

Scraping categories: ['Travel', 'Mystery', 'Historical Fiction', 'Sequential Art', 'Classics']
Scraped 163 books across 5 categories


category
Sequential Art        75
Mystery               32
Historical Fiction    26
Classics              19
Travel                11
Name: count, dtype: int64

## 2. Clean

Convert each raw field to its proper type, and handle rows that fail to
parse instead of letting the pipeline crash:

- **Numeric fields** (`price_gbp`, `rating`): if a value can't be parsed,
  impute the **median** of the successfully parsed values for that column.
  A single garbled number shouldn't sink the row, and the median is robust
  to outliers.
- **Non-numeric identity fields** (`title`, `in_stock`): if the title is
  blank or the availability text doesn't match a known pattern, we can't
  trust the row at all, so it is **dropped** rather than guessing a
  boolean/text value.

In [3]:
STAR_WORD_TO_INT = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def parse_price(raw):
    """'£51.77' -> 51.77; None if the text has no usable number."""
    try:
        return float(re.sub(r"[^\d.]", "", raw))
    except (TypeError, ValueError):
        return None


def parse_rating(word):
    """'Three' -> 3; None for any word we don't recognise."""
    return STAR_WORD_TO_INT.get(word)


def parse_availability(raw):
    """'In stock (19 available)' -> True; unrecognised text -> None."""
    text = (raw or "").strip().lower()
    if "in stock" in text:
        return True
    if "out of stock" in text:
        return False
    return None


df = df_raw.copy()
df["price_gbp"] = df["price"].apply(parse_price)
df["rating"] = df["star_rating"].apply(parse_rating)
df["in_stock"] = df["availability"].apply(parse_availability)

In [4]:
n_before = len(df)

# Numeric fields: median-impute anything that failed to parse.
for col in ["price_gbp", "rating"]:
    n_missing = int(df[col].isna().sum())
    if n_missing:
        median_val = df[col].median()
        print(f"Imputing {n_missing} missing '{col}' value(s) with median {median_val}")
        df[col] = df[col].fillna(median_val)
df["rating"] = df["rating"].round().astype(int)

# Non-numeric identity fields: drop rows we can't trust rather than guess.
missing_title = df["title"].isna() | (df["title"].str.strip() == "")
missing_stock = df["in_stock"].isna()
n_dropped = int((missing_title | missing_stock).sum())
if n_dropped:
    print(f"Dropping {n_dropped} row(s) with an unparseable title/availability")
df = df[~(missing_title | missing_stock)].reset_index(drop=True)
df["in_stock"] = df["in_stock"].astype(bool)

print(f"{n_before - len(df)} row(s) dropped, {len(df)} rows remain "
      f"(0 parse failures were encountered on this run's live scrape)")

0 row(s) dropped, 163 rows remain (0 parse failures were encountered on this run's live scrape)


## 3. Convert to INR

`price_inr = price_gbp * GBP_TO_INR`, using the fixed baseline rate declared
above (**1 GBP = 105.50 INR**) — no API call, no network lookup, no date
reference, exactly as the assignment requires for the graded path.

In [5]:
df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)
df[["title", "price_gbp", "price_inr"]].head()

,title,price_gbp,price_inr
0,It's Only the Himalayas,45.17,4765.44
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,5214.86
2,See America: A Celebration of Our National Par...,48.87,5155.78
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17
4,Under the Tuscan Sun,37.33,3938.31


## 4. Normalized SQLite schema

Two tables sharing a primary/foreign key relationship:

- `categories(category_id PK, category_name UNIQUE)`
- `books(book_id PK, title, price_gbp, price_inr, rating, in_stock, category_id REFERENCES categories)`

The schema is dropped and recreated from scratch on every run so re-running
this notebook never silently duplicates rows on top of a stale `books.db`.

In [6]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.executescript("""
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS categories;

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
);

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER REFERENCES categories(category_id)
);
""")
conn.commit()

In [7]:
cur.executemany(
    "INSERT INTO categories (category_name) VALUES (?)",
    [(c,) for c in df["category"].unique()],
)
conn.commit()

cat_id_map = dict(cur.execute("SELECT category_name, category_id FROM categories").fetchall())
df["category_id"] = df["category"].map(cat_id_map)

books_cols = ["title", "price_gbp", "price_inr", "rating", "in_stock", "category_id"]
books_data = df[books_cols].copy()
books_data["in_stock"] = books_data["in_stock"].astype(int)

cur.executemany(
    "INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id) "
    "VALUES (?, ?, ?, ?, ?, ?)",
    books_data.itertuples(index=False, name=None),
)
conn.commit()

pd.read_sql("SELECT COUNT(*) AS n_books FROM books", conn)

,n_books
0,163


## 5. SQL queries

Five queries against the loaded database, collectively covering
`SELECT`/`WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN`/`BETWEEN`, and a
`JOIN` across both tables. Each is read back with `pd.read_sql`.

In [8]:
# Q1 — SELECT / WHERE / ORDER BY / LIMIT: 10 priciest in-stock books (INR).
q1 = """
SELECT title, price_inr
FROM books
WHERE in_stock = 1
ORDER BY price_inr DESC
LIMIT 10
"""
result_q1 = pd.read_sql(q1, conn)
result_q1

,title,price_inr
0,Boar Island (Anna Pigeon #19),6275.14
1,Candide,6185.46
2,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,6087.35
3,El Deafo,6078.91
4,Animal Farm,6036.71
5,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",6019.83
6,A Year in Provence (Provence #1),6000.84
7,"Giant Days, Vol. 1 (Giant Days #1-4)",5988.18
8,The Past Never Ends,5960.75
9,The Last Painting of Sara de Vos,5860.52


In [9]:
# Q2 — DISTINCT: every star-rating value actually present in the data.
q2 = "SELECT DISTINCT rating FROM books ORDER BY rating"
result_q2 = pd.read_sql(q2, conn)
result_q2

,rating
0,1
1,2
2,3
3,4
4,5


In [10]:
# Q3 — IN: books rated 4 or 5 stars.
q3 = "SELECT title, rating FROM books WHERE rating IN (4, 5)"
result_q3 = pd.read_sql(q3, conn)
result_q3

,title,rating
0,Full Moon over Noah’s Ark: An Odyssey to Mount...,4
1,A Year in Provence (Provence #1),4
2,"1,000 Places to See Before You Die",5
3,Sharp Objects,4
4,The Past Never Ends,4
5,The Murder of Roger Ackroyd (Hercule Poirot #4),4
6,A Time of Torment (Charlie Parker #14),5
7,Murder at the 42nd Street Library (Raymond Amb...,4
8,What Happened on Beale Street (Secrets of the ...,5
9,The Bachelor Girl's Guide to Murder (Herringfo...,5


In [11]:
# Q4 — BETWEEN: books priced 20-30 GBP.
q4 = "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 30"
result_q4 = pd.read_sql(q4, conn)
result_q4

,title,price_gbp
0,The Road to Little Dribbling: Adventures of an...,23.21
1,"1,000 Places to See Before You Die",26.08
2,Poisonous (Max Revere Novels #3),26.80
3,The Widow,27.26
4,What Happened on Beale Street (Secrets of the ...,25.37
5,Delivering the Truth (Quaker Midwife Mystery #1),20.89
6,The Mysterious Affair at Styles (Hercule Poiro...,24.80
7,The Silkworm (Cormoran Strike #2),23.05
8,Extreme Prey (Lucas Davenport #26),25.40
9,Career of Evil (Cormoran Strike #3),24.72


In [12]:
# Q5 — JOIN (+ ORDER BY / LIMIT): 10 highest-rated books with their category name.
q5 = """
SELECT b.title, b.rating, c.category_name
FROM books b
JOIN categories c ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.title ASC
LIMIT 10
"""
result_q5 = pd.read_sql(q5, conn)
result_q5

,title,rating,category_name
0,"1,000 Places to See Before You Die",5,Travel
1,A Flight of Arrows (The Pathfinders #2),5,Historical Fiction
2,A Spy's Devotion (The Regency Spies of London #1),5,Historical Fiction
3,A Time of Torment (Charlie Parker #14),5,Mystery
4,Batman: The Dark Knight Returns (Batman),5,Sequential Art
5,Between Shades of Gray,5,Historical Fiction
6,"Bleach, Vol. 1: Strawberry and the Soul Reaper...",5,Sequential Art
7,El Deafo,5,Sequential Art
8,"Fruits Basket, Vol. 1 (Fruits Basket #1)",5,Sequential Art
9,"Fruits Basket, Vol. 2 (Fruits Basket #2)",5,Sequential Art


## 6. pandas vs. SQL cross-check

`result_q1` and `result_q3` above were already read back via `pd.read_sql`.
Now reproduce the Q5 join purely in pandas with `pd.merge` on the in-memory
DataFrames (no SQL), assert both approaches agree, and display the two
results side by side in a single table for a visual check.

In [13]:
categories_df = pd.read_sql("SELECT * FROM categories", conn)

result_merge = (
    df.merge(categories_df, on="category_id")[["title", "rating", "category_name"]]
    .sort_values(["rating", "title"], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)

assert result_merge.equals(result_q5), "pd.merge result should match the SQL JOIN result"
print("pd.merge output matches the SQL JOIN output.")

pd.merge output matches the SQL JOIN output.


In [14]:
# Side by side: pd.read_sql (SQL JOIN) on the left, pd.merge on the right — same 10 rows.
pd.concat({"pd.read_sql (SQL JOIN)": result_q5, "pd.merge": result_merge}, axis=1)

pd.read_sql (SQL JOIN)         \
                                               title rating   
0                 1,000 Places to See Before You Die      5   
1            A Flight of Arrows (The Pathfinders #2)      5   
2  A Spy's Devotion (The Regency Spies of London #1)      5   
3             A Time of Torment (Charlie Parker #14)      5   
4           Batman: The Dark Knight Returns (Batman)      5   
5                             Between Shades of Gray      5   
6  Bleach, Vol. 1: Strawberry and the Soul Reaper...      5   
7                                           El Deafo      5   
8           Fruits Basket, Vol. 1 (Fruits Basket #1)      5   
9           Fruits Basket, Vol. 2 (Fruits Basket #2)      5   

                                                                pd.merge  \
        category_name                                              title   
0              Travel                 1,000 Places to See Before You Die   
1  Historical Fiction            A Flight of Arrows (The Pathfinders #2)   
2  Historical Fiction  A Spy's Devotion (The Regency Spies of London #1)   
3             Mystery             A Time of Torment (Charlie Parker #14)   
4      Sequential Art           Batman: The Dark Knight Returns (Batman)   
5  Historical Fiction                             Between Shades of Gray   
6      Sequential Art  Bleach, Vol. 1: Strawberry and the Soul Reaper...   
7      Sequential Art                                           El Deafo   
8      Sequential Art           Fruits Basket, Vol. 1 (Fruits Basket #1)   
9      Sequential Art           Fruits Basket, Vol. 2 (Fruits Basket #2)   

                              
  rating       category_name  
0      5              Travel  
1      5  Historical Fiction  
2      5  Historical Fiction  
3      5             Mystery  
4      5      Sequential Art  
5      5  Historical Fiction  
6      5      Sequential Art  
7      5      Sequential Art  
8      5      Sequential Art  
9      5      Sequential Art

In [15]:
conn.close()